# 4: Causal Discovery

Traditional causal inference requires a pre-built causal model, often impractical to construct.
Causal discovery instead uses observational data to find causal links.

This chapter introduces computational techniques for causal discovery, outlining the assumptions underlying different algorithms.
It analyses their strengths and weaknesses before applying them to a real-world case study, demonstrating practical utility.

## 4.1: Assumptions of Causal Discovery

In modelling, a **forward problem** predicts outcomes from given inputs (e.g., estimating causal effects from a model).
An **inverse problem** identifies inputs from observed outcomes (e.g., discovering a causal model from data).
Solving inverse problems is harder, as multiple models may fit the same data (Maclaren & Nicholson, 2019).

Solving inverse problems typically requires assumptions to narrow solutions.
Four common assumptions are used in causal discovery algorithms (Verma & Pearl 2022; Frydenberg 1990).


### 4.1.1: Markov Assumption

The **Markov assumption** states that in causal graphs, each node is conditionally independent of its non-descendants given its parents.
Knowing a node's parents makes other variables irrelevant for predicting it, simplifying complex causal structures into conditional independence statements.

> 💡 Tip
>
> Under the Markov assumption, d-separated variables in graph G are independent in distribution P.

$$X \perp_G Y | Z \implies X \perp_P Y | Z$$


### 4.1.2: Faithfulness Assumptions

**Faithfulness** is the opposite of the Markov assumption: statistical independencies in $P$ imply d-separations in $G$.

$$X \perp_G Y | Z \impliedby X \perp_P Y | Z$$

Violations of the faithfulness assumption occur when observed statistical dependencies do not match d-separations in a causal graph.
For example, if two causal paths between nodes cancel each other, the nodes may appear statistically independent in the data despite not being d-separated.


### 4.1.3: Causal Sufficiency

**Causal sufficiency** assumes all relevant variables are measured, with no unmeasured confounders affecting both exposure and outcome.


### 4.1.4: Acyclicity

The **acyclicity assumption** in causal discovery requires no directed cycles (feedback loops) in the causal graph.


## 4.2: From Assumptions to Structures

**Markov equivalence** identifies graphs with identical conditional independencies, reducing structural search in causal models.

> 💡 Tip
>
> Two graphs are **Markov equivalent** if they share the same conditional independencies, enabling simplification of causal models.

**Chains** and **forks** (Chapter 3) share a Markov equivalence class, distinct from the immorality structure.
Under Markov assumptions, all chains and forks share a standard skeleton: the graph's undirected version.

Two key graph properties for distinguishing structures are immoralities and the skeleton, underpinning a key theorem.

_Two graphs are Markov equivalent if they share the same skeleton and immoralities._

We discover partially directed graphs from data by analysing conditional dependencies to build a skeleton and direct edges using immoralities.


## 4.3: Causal Discovery Algorithms

This section examines how causal discovery algorithms find causal links in observational or experimental data.


### 4.3.1: Constraint-Based Algorithms

Constraint-based causal discovery uses conditional independence tests, which can be difficult when variable dependence is unclear.
While applicable across many domains, effectiveness depends on large sample sizes for accurate tests.
A key drawback is reliance on the faithfulness assumption, which is often hard to satisfy in practice.


#### 4.3.1.1: PC Algorithm

The **PC algorithm** (Spirtes et al. 2000) is an early causal discovery method relying on the causal Markov condition, faithfulness, and no latent confounders.

> PC may stand for Partial Correlation (concept used in the algorithm) but actually references the initial letters of the names of the two main authors (Peter Spirtes and ClarkGlymour).

Consider an example with a known causal structure.
The PC algorithm reconstructs this structure, as shown in [Figure 4.1](#fig-pc).

<center>
  <img
    src="images/pc_algorithm_steps.png"
    alt="PC algorithms and steps"
    width="400"/>
  <a id="fig-pc"></a>
  <h6>Figure 4.1: PC algorithms and steps</h6>
</center>

The steps of the PC algorithm are:

1. Create a complete undirected, fully connected graph on all the variables.
2. Remove all the independent edges from the observational data.
   For example, remove the edge $X \longleftrightarrow Y$ because $X \perp Y$.
3. Remove any edge $X \longleftrightarrow Y$ if $A$ and $B$ are conditionally independent given some $C$ connected to $A$ or $b$.
  For example, remove the edges $X \longleftrightarrow W$ and $Y \longleftrightarrow W$ since $X \perp W|_Y$ and $Y \perp W|_Z$.
4. A v-structure occurs when $A \longleftrightarrow B$, $A \longleftrightarrow C$, and not $A \longleftrightarrow C$.
   Provided that $B$ wasn't part of the conditioning set that made $A$ and $C$ independent, replace the edges in the v-structure with $A \longrightarrow B \longleftarrow C$.
   For example, removing $X \longleftrightarrow Y$ that has no conditioning on $Z$ leaves behind a v-structure that can be replaced with $X \longrightarrow Z \longleftarrow Y$.
5. Set $B \longrightarrow C$ in a triple $A \longrightarrow B \longleftrightarrow C$ if $A$ and $C$ are non-adjacent.
   For example, orienting $Y \longrightarrow Z \longleftrightarrow W$ as $Y \longrightarrow Z \longrightarrow W$ uniquely recovers the structure.

> 💡 Tip
>
> The PC algorithm converges to the true Markov equivalence class with large samples if conditional independence decisions are accurate.


#### 4.3.1.2 FCI Algorithm

*Fast Causal Inference (FCI)* is a constraint-based algorithm capable of identifying unknown confounders that is more efficient and scalable than the PC algorithm (Spirtes et al., 2000).

FCI and PC algorithms are similar: both start with a fully connected undirected graph and remove edges between conditionally independent variables.

The algorithm first removes edges based on conditional independence.
It then orients the remaining edges where possible, typically using collider structures (e.g., $X \longrightarrow Y \longleftarrow Z$ when $X$ and $Z$ are unconditionally independent).
Undirected edges (marked by circles at nodes) indicate unknown directionality, signifying unobserved confounding or indeterminate relationships between variables.

<center>
  <img
    src="images/fci.png"
    alt="FCI Algorithm"
    width="400"/>
  <a id="fig-fci"></a>
  <h6>Figure 4.2: FCI Algorithm</h6>
</center>

Spirtes et al. introduced Anytime FCI (AFCI, 2001), a variant of FCI that uses smaller conditioning sets than a fixed $K$ threshold.

*Really Fast Causal Inference (RFCI)* is faster than FCI as it uses fewer conditional independence tests on smaller variable subsets.
It is more reliable with small samples, avoiding power issues in high-order tests.
However, this reduces result informativeness.

Current Causal Discovery (CCD) and SAT-based methods remove acyclicity and causal sufficiency assumptions (Richardson 1996; Hyttinen et al. 2013).


### 4.3.2 Score-Based Algorithms

Score-based algorithms start with an undirected graph and iteratively adjust edges to maximise a score.
Common scores include **Bayesian Information Criterion (BIC)**, **Akaike Information Criterion (AIC)**, and **Maximum Likelihood (ML)**, which balance model complexity against fit to prevent overfitting and ensure reliable causal inferences.


#### 4.3.2.1 Greedy Equivalence Search Algorithm

The **Greedy Equivalence Search (GES)** is a causal discovery algorithm that builds a partially directed graph from data.
It starts with an empty graph and iteratively adds edges using score functions (like BIC) to maximise fit.
GES uses a greedy approach to select edges improving the score, plus backtracking to correct edge directions.

The **Greedy Fast Causal Inference (GFCI)** algorithm (Ogarrio et al., 2016) combines GES and FCI for causal discovery.
GES generates a supergraph skeleton, which FCI then prunes and orients correctly.

Fast **Greedy Equivalence Search (FGES)** combines GES and FCI, using a parallelised GES variant and v-structures to determine edge directions.


### 4.3.3 Semi-Parametric Algorithms

Causal inference methods face three key limitations:

1. They rely on the faithfulness assumption, which may not hold.
2. They often require large sample sizes for accurate conditional independence testing.
3. They can only identify Markov equivalent causal structures, failing to distinguish between models with identical conditional independence patterns.

Based on Geiger and Pearl (1990) and Meek (2013), multinomial or linear Gaussian models only identify graphs up to Markov equivalence.
However, under **linear non-Gaussian noise** or **nonlinear additive noise settings** (with semi-parametric assumptions), the causal graph becomes fully identifiable.


### 4.3.3.1 Linear Non-Gaussian Noise (LiNGAM)

In linear non-Gaussian noise settings, structural equations take this form:

$$Y := f(X) + U$$

with linear $f(X)$, $X \perp U$, and unobserved non-Gaussian $U$ (Shimizu et al. 2006).

Darmois and Skitovich (1953, 1954) established identifiability in linear non-Gaussian systems.
Specifically, if the true structural causal model is $Y := f(X) + U$ with $X \perp U$, no reverse model $X = f(Y) + \tilde{U}$ with $Y \perp \tilde{U}$ can reproduce the joint distribution $P(X,Y)$.

In non-linear Gaussian models, the causal direction $X \longrightarrow Y$ is identified when residuals are independent of the input.
Conversely, for the anti-causal direction $X \longleftarrow Y$, residuals depend on the input.

Extensions have relaxed key assumptions: Shimizu et al. (multivariate), Hoyer et al. (causal sufficiency), and Lacerda et al. (acyclicity, 2012).


#### 4.3.3.2 Nonlinear Additive Noise

Nonlinear additive noise models identify causal graphs by assuming nonlinear causal mechanisms with additive noise (Hoyer et al., 2008).

$$\forall_i X_i := f(pa_i) + U_i$$

where $f$ is nonlinear and $pa_i$ are $X$'s parents.

Zhang and Hyvarinen (2012) extended the model by replacing nonlinear additive noise with a post-nonlinear transformation applied after noise addition.

$$\forall_i X_i := g(f(pa_i) + U_i) for all i,$$

with X independent of U ($X \perp U$).
